In [2]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(["science", "no-latex"])
from pymatgen.core import Structure
from pymatgen.analysis.structure_matcher import StructureMatcher

# load csvs recording unique ones in experimental structures and unique ones in theoretical ones

In [ ]:
data_positive = pd.read_csv(
    "all_structures/unique_experimental/unique_experimental.csv"
)
data_un_label = pd.read_csv("all_structures/unique_theoretical/unique_theoretical.csv")

un_label_index_to_drop = []
for index, row in data_un_label.iterrows():
    sub_data_positive = data_positive.loc[
        (data_positive["group_number"] == row["group_number"])
        & (data_positive["compositions"] == row["compositions"])
    ]
    the_struct = Structure.from_file(
        f"all_structures/unique_theoretical/{row.name}.vasp"
    )
    for index2, row2 in sub_data_positive.iterrows():
        if os.path.exists(f"all_structures/unique_experimental/{row2.name}.vasp"):
            exp_struct = Structure.from_file(
                f"all_structures/unique_experimental/{row2.name}.vasp"
            )
        else:
            exp_struct = Structure.from_file(
                f"all_structures/unique_experimental/{row2.name}.cif"
            )
        sm = StructureMatcher()
        if sm.fit(the_struct, exp_struct):
            un_label_index_to_drop.append(index)
            break

data_un_label.drop(data_un_label.index[un_label_index_to_drop], inplace=True)
data_un_label.to_csv("all_structures/unique_theoretical/unique_true_theoretical.csv")

# Split structures into training and evaluation set

In [ ]:
data_positive = pd.read_csv(
    "all_structures/unique_experimental/unique_experimental.csv"
)
data_un_label = pd.read_csv(
    "all_structures/unique_theoretical/unique_true_theoretical.csv"
)

random_permutation_positive = np.random.permutation(len(data_positive))
n_train_positive = int(0.8 * len(data_positive))
random_permutation_un_label = np.random.permutation(len(data_un_label))
n_train_un_label = int(0.8 * len(data_un_label))

data_positive_train = data_positive.iloc[random_permutation_positive[:n_train_positive]]
data_positive_evaluation = data_positive.iloc[
    random_permutation_positive[n_train_positive:]
]
data_un_label_train = data_un_label.iloc[random_permutation_un_label[:n_train_un_label]]
data_un_label_evaluation = data_un_label.iloc[
    random_permutation_un_label[n_train_un_label:]
]

train = pd.concat([data_positive_train, data_un_label_train])
train.insert(
    1,
    "synthesizability",
    [1] * len(data_positive_train) + [0] * len(data_un_label_train),
)
train[:, :2].to_csv("../train/train_set.csv", index=False, header=False)

evaluate = pd.concat([data_positive_evaluation, data_un_label_evaluation])
evaluate.insert(
    1,
    "synthesizability",
    [1] * len(data_positive_evaluation) + [0] * len(data_un_label_evaluation),
)
evaluate[:, :2].to_csv("../evaluation/evaluation_set.csv", index=False, header=False)

# copy cif_files

In [ ]:
from pymatgen.io.cif import CifWriter

# train
train = pd.read_csv("../train/train_set.csv", header=None)
names = train.iloc[:, 0].to_list()
synthesizability = train.iloc[:, 1].to_list()
for nn, ss in zip(names, synthesizability):
    if ss == 1:
        if os.path.exists(f"all_structures/unique_experimental/{nn}.vasp"):
            struct = Structure.from_file(
                f"all_structures/unique_experimental/{nn}.vasp"
            )
        else:
            struct = Structure.from_file(f"all_structures/unique_experimental/{nn}.cif")
    else:
        struct = Structure.from_file(f"all_structures/unique_theoretical/{nn}.vasp")
    CifWriter(struct, symprec=0.01, angle_tolerance=5).write_file(
        f"../train/cif_files/{nn}.cif"
    )


# evaluation
evaluation = pd.read_csv("../evaluation/evaluation_set.csv", header=None)
names = train.iloc[:, 0].to_list()
synthesizability = train.iloc[:, 1].to_list()
for nn, ss in zip(names, synthesizability):
    if ss == 1:
        if os.path.exists(f"all_structures/unique_experimental/{nn}.vasp"):
            struct = Structure.from_file(
                f"all_structures/unique_experimental/{nn}.vasp"
            )
        else:
            struct = Structure.from_file(f"all_structures/unique_experimental/{nn}.cif")
    else:
        struct = Structure.from_file(f"all_structures/unique_theoretical/{nn}.vasp")
    CifWriter(struct, symprec=0.01, angle_tolerance=5).write_file(
        f"../evaluation/cif_files/{nn}.cif"
    )